In [10]:
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import cv2

## Preprocess images

In [11]:



# Folder paths
up_folder = "bridgeUp"
down_folder = "bridgeDown"

# Store features and labels
X = []
y = []

# Use the same image dimensions for every image
image_size = (1080, 1080)

# Load bridge-up images (label 1)
for filename in os.listdir(up_folder):
    filepath = os.path.join(up_folder, filename)

    try:
        image = Image.open(filepath).convert("L")
        image = image.resize(image_size)
        image = np.array(image)

        features = hog(
            image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )

        X.append(features)
        y.append(1)

    except (OSError, ValueError):
        continue

# Load bridge-down images (label 0)
for filename in os.listdir(down_folder):
    filepath = os.path.join(down_folder, filename)

    try:
        image = Image.open(filepath).convert("L")
        image = image.resize(image_size)
        image = np.array(image)

        features = hog(
            image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )

        X.append(features)
        y.append(0)

    except (OSError, ValueError):
        continue

# Convert to NumPy arrays
X = np.array(X)
y = np.array(y)

print("Number of images:", len(y))
print("Feature matrix shape:", X.shape)
print("Label array shape:", y.shape)
print("Up images:", np.sum(y == 1))
print("Down images:", np.sum(y == 0))


Number of images: 117
Feature matrix shape: (117, 646416)
Label array shape: (117,)
Up images: 50
Down images: 67


## Split into training and testing sets

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

Training images: 93
Testing images: 24


In [13]:
model = SVC(C = 1, kernel = "linear")

model.fit(X_train, y_train)

SVC(C=1, kernel='linear')

## Use testing data on model

In [14]:
y_train_pred = model.predict(X_train)

training_error = np.mean(y_train_pred != y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

testing_error = np.mean(y_pred != y_test)
print("Testing Error: ", testing_error)
print("Training Error: ",training_error )

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Bridge Down", "Bridge Up"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0
Testing Error:  0.0
Training Error:  0.0

Classification Report:
              precision    recall  f1-score   support

 Bridge Down       1.00      1.00      1.00        14
   Bridge Up       1.00      1.00      1.00        10

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24


Confusion Matrix:
[[14  0]
 [ 0 10]]


## Predict on new image

In [15]:
# Path to the new image
image_path = "allTheWayDown3.png" #PUT IMAGE HERE

# Load and preprocess the image
image = Image.open(image_path).convert("L")
image = image.resize(image_size)
image = np.array(image)

# Extract HOG features using the same settings
features = hog(
    image,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys"
)

# Convert features into the format expected by the model
features = features.reshape(1, -1)

# Predict the bridge position
prediction = model.predict(features)[0]

# Display the result
if prediction == 1:
    print("Prediction: Bridge Up")
else:
    print("Prediction: Bridge Down")

Prediction: Bridge Down


## Predict on Video Footage

In [21]:
# Path to the video
video_path = "test5Trim.mp4"

# Open the video
cap = cv2.VideoCapture(video_path)


while cap.isOpened():

    # Read the next frame
    ret, frame = cap.read()

    if not ret:

        break

    # Resize frame to match training images
    frame = cv2.resize(frame, image_size)

    # Convert frame to grayscale
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Extract HOG features using the same settings as training
    features = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    # Reshape features for prediction
    features = features.reshape(1, -1)

    # Predict bridge position
    prediction = model.predict(features)[0]

    # Display prediction
    if prediction == 1:
        label = "Bridge Up"
        color = (0, 255, 0)
    else:
        label = "Bridge Down"
        color = (0, 0, 255)

    # Add prediction to frame
    cv2.putText(
        frame,
        label,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2
    )

    # Display video
    cv2.imshow("Bridge Position Prediction", frame)

    # Press q to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release video
cap.release()
cv2.destroyAllWindows()